# 04 — Single-line-to-ground fault

**Goal:** run the bundled SLG fault demonstrator, inspect the faulted network state, read the current, then verify the saved evidence.

**Teaching inputs:** bus 675, phase A, rf = 0.001 ohm.

**Prediction:** the fault current should be positive and finite for the declared source and feeder model.

Run the numbered cells in order. The direct OpenDSS section at the end is optional.

In [1]:
#@title 1. Setup — run once
import contextlib, io, urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
try:
    _blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
    _digest = hashlib.sha256(_blob).hexdigest()
    if _digest != _HELPER_SHA256:
        raise ValueError(f"lesson helper hash mismatch (expected {_HELPER_SHA256}, got {_digest})")
    _helper_output = io.StringIO()
    with contextlib.redirect_stdout(_helper_output):
        exec(compile(_blob, "lesson helper", "exec"))
except Exception as exc:
    print("What happened: the pinned lesson helper could not be downloaded or verified.")
    print(f"Details: {type(exc).__name__}: {exc}")
    print("Next: check network access to the public helper URL, then rerun this setup cell.")
    raise SystemExit(1) from None
for _line in _helper_output.getvalue().splitlines():
    if "lesson helpers ready" not in _line.lower():
        print(_line)
print("Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.")

# Compatibility renderer for the currently pinned public wheel; newer wheels use cept.public_notebook.
def display_run_compat(run_dir):
    import html as _html
    from IPython.display import HTML, display
    result = read(Path(run_dir) / 'results.json')
    sld = result.get('sld') or result.get('sld_after') or {}
    nodes, edges = sld.get('nodes') or [], sld.get('edges') or []
    if not nodes:
        display(HTML('<p><strong>CEPT result:</strong> this run does not carry an SLD.</p>'))
        return
    xs, ys = [float(n['x']) for n in nodes], [float(n['y']) for n in nodes]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    dx, dy = max(x1-x0, 1.0), max(y1-y0, 1.0)
    def xy(node):
        return 65 + (float(node['x'])-x0)/dx*870, 45 + (float(node['y'])-y0)/dy*420
    pos = {str(n['id']): xy(n) for n in nodes}
    line_svg = []
    for edge in edges:
        if str(edge.get('src')) in pos and str(edge.get('dst')) in pos:
            a, b = pos[str(edge['src'])], pos[str(edge['dst'])]
            dash = ' stroke-dasharray=\"8 6\"' if edge.get('status') == 'open' else ''
            edge_label = _html.escape(str(edge.get('id', 'branch')))
            line_svg.append(f'<line x1=\"{a[0]:.1f}\" y1=\"{a[1]:.1f}\" x2=\"{b[0]:.1f}\" y2=\"{b[1]:.1f}\" stroke=\"#7a879a\" stroke-width=\"3\"{dash}><title>{edge_label}</title></line>')
    vmin, vmax = float(sld.get('v_min_pu', .95)), float(sld.get('v_max_pu', 1.05))
    bus_svg, rows = [], []
    for node in nodes:
        volts = {int(k): float(v) for k, v in (node.get('v_pu') or {}).items()}
        angles = {int(k): float(v) for k, v in (node.get('angle_deg') or {}).items()}
        values = list(volts.values())
        low, high = any(v < vmin for v in values), any(v > vmax for v in values)
        status = 'NO DATA' if not values else 'OUT' if low and high else 'UNDER' if low else 'OVER' if high else 'OK'
        fill = {'OK':'#e8f5ec','UNDER':'#fff3d9','OVER':'#ffe7e1','OUT':'#f7e7ff','NO DATA':'#eef1f5'}[status]
        stroke = {'OK':'#2f7d4a','UNDER':'#a46700','OVER':'#b8432e','OUT':'#8147a6','NO DATA':'#7b8796'}[status]
        x, y = pos[str(node['id'])]
        phase = lambda p: '—' if p not in volts else f'{volts[p]:.4f} pu' + (f' @ {angles[p]:.2f}°' if p in angles else '')
        tip = _html.escape('Bus '+str(node['id'])+'\nStatus: '+status+'\n'+'\n'.join(f'{label}: {phase(p)}' for p,label in [(1,'A'),(2,'B'),(3,'C')] if p in volts))
        label = _html.escape(str(node['id']))
        bus_svg.append(f'<g tabindex=\"0\"><title>{tip}</title><rect x=\"{x-31:.1f}\" y=\"{y-11:.1f}\" width=\"62\" height=\"22\" rx=\"5\" fill=\"{fill}\" stroke=\"{stroke}\" stroke-width=\"2\"/><text x=\"{x:.1f}\" y=\"{y+4:.1f}\" text-anchor=\"middle\" font-size=\"12\" font-weight=\"700\">{label}</text></g>')
        rows.append('<tr><th>'+label+'</th><td>'+phase(1)+'</td><td>'+phase(2)+'</td><td>'+phase(3)+'</td><td><strong>'+status+'</strong></td></tr>')
    display(HTML('<div style=\"font-family:system-ui,sans-serif\"><h3>Interactive CEPT SLD</h3><p style=\"color:#657187\">Hover or focus a bus for solver-returned values.</p><div style=\"overflow:hidden;border:1px solid #d9dee8;border-radius:10px\"><svg viewBox=\"0 0 1000 510\" style=\"width:100%;height:auto;display:block\">'+''.join(line_svg)+''.join(bus_svg)+'</svg></div><div style=\"overflow-x:auto;margin-top:10px\"><table style=\"border-collapse:collapse;width:100%;min-width:650px\"><thead><tr><th>Bus</th><th>Phase A</th><th>Phase B</th><th>Phase C</th><th>Status</th></tr></thead><tbody>'+''.join(rows)+'</tbody></table></div></div>'))


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version
cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


In [2]:
#@title 2. Inputs — fault location and impedance
RUN_DIR = WORKSPACE / "runs" / "04-fault-study"
FAULT_BUS = "675"
FAULT_PHASE = 1
FAULT_RESISTANCE_OHM = 0.001
table(
    ["declared input", "value", "unit"],
    [
        ("network", "IEEE 13-node feeder", "text"),
        ("fault bus", FAULT_BUS, "bus"),
        ("fault phase", "A", "phase"),
        ("fault resistance", FAULT_RESISTANCE_OHM, "ohm"),
    ],
)

| declared input | value | unit |
| --- | --- | --- |
| network | IEEE 13-node feeder | text |
| fault bus | 675 | bus |
| fault phase | A | phase |
| fault resistance | 0.001 | ohm |


In [3]:
# 3. Run study — same CEPT CLI as a normal terminal
!cept study demo fault \
    --network ieee13 \
    --out runs/04-fault-study \
    --force \
    --format text

CEPT study result: FINISHED
----------------------------
Result             Finished the fault study and saved the evidence
Saved run          runs\04-fault-study
Case fingerprint   44d3766e5e8c (matches the case you ran)

What this means
  The study completed and saved solver-backed evidence.
  It does NOT approve a real project or field installation.

Next
  cept study verify runs\04-fault-study --format text


In [4]:
#@title 4. Explore — SLD and bus status
RUN_DIR = WORKSPACE / "runs" / "04-fault-study"
try:
    from cept.public_notebook import display_run
except ModuleNotFoundError:
    display_run = display_run_compat
display_run(RUN_DIR)

Bus,Phase A,Phase B,Phase C,Status
611,—,—,1.2235 pu @ 132.01°,OVER
632,0.5915 pu @ -2.77°,1.1656 pu @ -131.51°,1.1292 pu @ 126.18°,OUT
633,0.5904 pu @ -2.79°,1.1627 pu @ -131.54°,1.1267 pu @ 126.13°,OUT
634,0.5763 pu @ -3.48°,1.1432 pu @ -131.97°,1.1078 pu @ 125.70°,OUT
645,—,1.1562 pu @ -131.70°,1.1272 pu @ 126.21°,OVER
646,—,1.1547 pu @ -131.78°,1.1253 pu @ 126.26°,OVER
650,0.9988 pu @ -0.02°,0.9998 pu @ -119.99°,0.9998 pu @ 119.97°,OK
652,0.1054 pu @ -37.66°,—,—,UNDER
670,0.4243 pu @ -5.48°,1.2218 pu @ -134.46°,1.1593 pu @ 128.24°,OUT
671,0.1058 pu @ -38.18°,1.3467 pu @ -139.55°,1.2279 pu @ 132.28°,OUT


In [5]:
#@title 5. Engineering result — fault current
results = read(RUN_DIR / "results.json")
fault = results["fault"]
table(
    ["quantity", "value", "unit"],
    [
        ("fault bus", fault["bus"], "bus"),
        ("fault type", fault["fault_type"], "text"),
        ("fault resistance", fault["rf_ohm"], "ohm"),
        ("fault phases", fault["phases"], "phase"),
        ("total fault current", fault["total_fault_current_a"], "A"),
    ],
)
assert fault["bus"].lower() == FAULT_BUS.lower()
assert fault["phases"] == [FAULT_PHASE]

| quantity | value | unit |
| --- | --- | --- |
| fault bus | 675 | bus |
| fault type | slg | text |
| fault resistance | 0.001 | ohm |
| fault phases | [1] | phase |
| total fault current | 2950.7 | A |


In [6]:
# 6. Verify — check this exact saved run
!cept study verify runs/04-fault-study --format text

CEPT study check: PASSED
----------------------------
Study              Fault study (OpenDSS)
Case fingerprint   44d3766e5e8c (matches the case you ran)

Checked   3 groups, 13 checks, all passed
  [PASS] Case identity (4 checks)
  [PASS] Solver result (2 checks)
  [PASS] Saved evidence (7 checks)

What this means
  The saved result matches its Case, solver run, and saved evidence.
  It does NOT approve a real project or field installation.

Saved evidence     runs\04-fault-study\public-verification.json

For the full check list
  cept study verify runs\04-fault-study --format json


## 7. Interpret

The reported current belongs to the declared source, feeder, fault location, phase, and resistance.

**What this proves:** the solver-backed fault result was persisted and its workflow evidence can be verified.

**What this does not prove:** breaker interrupting duty, relay settings, arc-flash results, or field short-circuit acceptance.

**Try next:** predict what should happen if fault resistance increases, then compare with a deliberately changed direct-solver calculation.

## Optional — direct OpenDSS comparison

The cells below solve the same declared fault directly in OpenDSS and compare the returned fault-element current with the persisted CEPT result.

In [7]:
#@title Under the hood - direct OpenDSS fault solve (optional)
MASTER_DSS = ieee13_master()
import opendssdirect as dss

dss.Basic.ClearAll()
dss.Basic.DataPath(str(MASTER_DSS.parent))
dss.Text.Command(f'Redirect "{MASTER_DSS}"')
dss.Text.Command(f'New Fault.lesson_fault Bus1={FAULT_BUS}.{FAULT_PHASE} phases=1 r={FAULT_RESISTANCE_OHM}')
dss.Text.Command('Solve')
assert dss.Solution.Converged()
os.chdir(WORKSPACE)
print("Direct OpenDSS fault solve finished: the faulted feeder converged.")


Direct OpenDSS fault solve finished: the faulted feeder converged.


In [8]:
#@title Under the hood — direct fault readback (optional)
dss.Circuit.SetActiveElement('Fault.lesson_fault')
currents = dss.CktElement.Currents()
direct_current_a = abs(complex(currents[0], currents[1]))
table(['source', 'bus', 'phase', 'fault resistance', 'current', 'units'], [('direct OpenDSS', FAULT_BUS, FAULT_PHASE, FAULT_RESISTANCE_OHM, direct_current_a, 'ohm / A')])
assert direct_current_a > 0

| source | bus | phase | fault resistance | current | units |
| --- | --- | --- | --- | --- | --- |
| direct OpenDSS | 675 | 1 | 0.001 | 2950.698148621237 | ohm / A |


In [9]:
#@title Compare solver outputs (optional details)
RUN_DIR = WORKSPACE / "runs" / "04-fault-study"
results = read(RUN_DIR / "results.json")
verify_summary = read(RUN_DIR / "public-verification.json")
fault = results["fault"]
cept_current_a = float(fault["total_fault_current_a"])
fault_abs_diff_a = abs(direct_current_a - cept_current_a)
table(
    ["source", "total fault current", "unit"],
    [("direct OpenDSS", direct_current_a, "A"), ("CEPT results.json", cept_current_a, "A")],
)
print()
print("Direct OpenDSS and CEPT fault currents agree")
print("-------------------------------------------")
print(f"Result        {'PASSED' if verify_summary['passed'] else 'Needs attention'}")
print(f"Fault current is about {cept_current_a:.1f} A")
print(f"The two solvers differ by {fault_abs_diff_a:.2f} A (limit 1.0 A)")
assert verify_summary["status"] == "PASS"
assert verify_summary["passed"] is True
assert fault["bus"].lower() == FAULT_BUS.lower() and fault["fault_type"] == "slg"
assert fault["phases"] == [FAULT_PHASE]
assert fault_abs_diff_a < 1.0


| source | total fault current | unit |
| --- | --- | --- |
| direct OpenDSS | 2950.698148621237 | A |
| CEPT results.json | 2950.7 | A |

Direct OpenDSS and CEPT fault currents agree
-------------------------------------------
Result        PASSED
Fault current is about 2950.7 A
The two solvers differ by 0.00 A (limit 1.0 A)
